# Diabetic readmission pipeline — Kaggle runner

**Inputs to attach** (Notebook → *Add Input*):
1. The dataset **Diabetes 130-US hospitals** (e.g. Kaggle dataset `brandao/diabetes`, or upload `diabetic_data.csv` + `IDs_mapping.csv` as your own dataset).
2. The code: upload the `code/` folder as a Kaggle dataset (e.g. named `readmit-code`).

The runner finds both automatically under `/kaggle/input`. Outputs go to `/kaggle/working/readmit_outputs` (download from the *Output* tab).
Internet is only needed if `duckdb` is missing — otherwise the pipeline falls back to SQLite.

In [ ]:
import importlib, subprocess, sys
for pkg in ("duckdb",):
    if importlib.util.find_spec(pkg) is None:
        try:
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)
        except Exception as e:
            print(f"{pkg} not installed ({e}); SQLite fallback will be used")

In [ ]:
import shutil
from pathlib import Path
src = next(Path('/kaggle/input').rglob('run_pipeline.py')).parent
dst = Path('/kaggle/working/code')
if dst.exists():
    shutil.rmtree(dst)
shutil.copytree(src, dst)
print('code copied from', src)
data = next(Path('/kaggle/input').rglob('diabetic_data.csv')).parent
print('data found at', data)

## 1 · Smoke test (≈1 minute) — validates every stage on a patient sample

In [ ]:
!cd /kaggle/working/code && python run_pipeline.py --smoke

## 2 · Full run (all stages)
Add `--mixed` to also fit the random-intercept model (slow), `--mice-m 10` for more imputations.

In [ ]:
!cd /kaggle/working/code && python run_pipeline.py

## 3 · Inspect outputs

In [ ]:
import pandas as pd
out = Path('/kaggle/working/readmit_outputs')
display(pd.read_csv(out/'tables/cohort_flow.csv'))
display(pd.read_csv(out/'tables/dq_checks.csv').query('not passed'))
display(pd.read_csv(out/'tables/model_exposure_summary.csv'))

In [ ]:
# Render the dashboard inline (also saved as readmit_outputs/dashboard.html)
from IPython.display import HTML
HTML((out/'dashboard.html').read_text(encoding='utf-8'))

In [ ]:
shutil.make_archive('/kaggle/working/readmit_outputs', 'zip', out)
print('Download readmit_outputs.zip from the Output tab')